In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

# CTOI catalog

In [ ]:
from wakai.catalog import get_ctois

ctois = get_ctois(clobber=True)
# tois.columns.tolist()

In [ ]:
ctois.tail()

In [ ]:
ctois[ctois.Notes.apply(lambda x: 'young' in str(x))]

In [ ]:
ctois.shape

# exominer++ PCs

https://zenodo.org/records/15466293

In [ ]:
import pandas as pd

# df_exominer = pd.read_csv('./exominerplusplus_catalog_labeled_tces_s1-s67_tess-spoc-2min_complete_1-14-2025_1039.csv')
df_exominer = pd.read_csv('../data/exominerplusplus_catalog_unk_tces_s1-s67_tess-spoc-2min_complete_1-16-2025_1014.csv')
df_exominer.tail()

In [ ]:
idx=ctois['TIC ID'].astype(int).isin([df_exominer['TIC ID'].astype(int)])
idx.sum()

# cross-match CTOI with Gaia DR3 using `astroquery.xmatch`

In [ ]:
import astropy.units as u
from astroquery.xmatch import XMatch
import pandas as pd

gaia_table = XMatch.query(
    cat1=open('../data/CTOIs.csv'),
    cat2="vizier:I/355/gaiadr3",  # The Vizier identifier for Gaia DR3
    max_distance=1 * u.arcsec,    # Set the maximum matching radius
    colRA1="RA",  # Column name for Right Ascension in TIC results
    colDec1="Dec" # Column name for Declination in TIC results
)
gaia_df = gaia_table.to_pandas()

# results to 6237 matches
matches = gaia_df.copy()

# Apply magnitude filtering criteria
# For main-sequence stars, the difference between TESS T and Gaia G is around 0–0.5 magnitudes
# Redder stars might have T - G values closer to 1–2 magnitudes.
# For giants or highly reddened stars, the difference could be larger.
if False:
    # results to 5971 matches
    matches = []
    for i, row in gaia_df.iterrows():
        teff=row['Stellar Eff Temp (K)']
        if teff<3500:
            magnitude_threshold = 2
        elif (teff>3500) and (teff<6000):
            magnitude_threshold = 1
        else:
            magnitude_threshold = 1.5
        if abs(row['Gmag'] - row['TESS Mag']) < magnitude_threshold:
            matches.append(row)
    matches = pd.concat(matches, axis=1).T
if False:
    # results to 5700 matches
    magnitude_threshold = 1
    matches = gaia_df[abs(gaia_df['Gmag'] - gaia_df['TESS Mag']) < magnitude_threshold]
if False:
    # results to 4878 matches
    teff_threshold = 1000
    matches = gaia_df[abs(gaia_df['Teff'] - gaia_df['Stellar Eff Temp (K)']) < teff_threshold]
matches.tail()

In [ ]:
matches.shape

In [ ]:
matches['GaiaDR3'] = matches['DR3Name'].apply(lambda x: int(x.split()[-1]))
matches.to_csv('../data/CTOIs_GaiaDR3_xmatch.csv', index=False)

In [ ]:
import pandas as pd

#https://zenodo.org/records/8250524
df = pd.read_csv('../data/Perren2023/UCC_cat.csv.gz')
df.tail()

In [ ]:
df_mem = pd.read_parquet('../data/Perren2023/UCC_members.parquet.gz')
df_mem.tail()

In [ ]:
matches['Source'] = matches.GaiaDR3.copy()

In [ ]:
idx1 = df_mem.Source.isin(matches.Source)
idx.sum()

In [ ]:
idx2 = matches.GaiaDR3.isin(df_mem.Source)
idx.sum()

In [ ]:
matches[idx2]['Promoted to TOI'].unique()

In [ ]:
final = pd.merge(df_mem[idx1], matches[idx2], on='Source')
final.tail()

In [ ]:
final.to_csv('../../toi-viz/data/CTOI/ctoi_in_Perren2023.csv', index=False)

In [ ]:
matches[idx2]['TIC ID'].to_csv('../../toi-viz/data/CTOI/ctoi_in_Perren2023_ticids.txt', 
                               index=False, header=False)